In [32]:
import os
drive_path='../../Drive/MyDrive'
os.listdir(drive_path)


['aae', 'ch10', 'ch9', 'ENCI', 'GAN', 'GAN - Copy']

In [31]:
from sklearn.datasets import fetch_20newsgroups

In [2]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']
twenty_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42
  )

In [3]:
len(twenty_train.filenames)

2257

In [4]:
print("\n".join(twenty_train.data[0].split("\n")[:3]))

From: sd345@city.ac.uk (Michael Collier)
Subject: Converting images to HP LaserJet III?
Nntp-Posting-Host: hampton


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.ensemble import RandomForestClassifier

text_clf = Pipeline([
  ('vect', CountVectorizer()),
  ('tfidf', TfidfTransformer()),
  ('clf', RandomForestClassifier()),
])

In [6]:
text_clf.fit(twenty_train.data, twenty_train.target)

,steps,"[('vect', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [7]:
import numpy as np
twenty_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42
)
predicted = text_clf.predict(twenty_test.data)
np.mean(predicted == twenty_test.target)

0.8082556591211718

In [33]:
import os
drive_path='../../Drive/MyDrive'
os.chdir(drive_path)
os.mkdir(drive_path+'/ch10/embeddings')
os.listdir(drive_path)

['aae', 'ch10', 'ch9', 'ENCI', 'GAN', 'GAN - Copy']

In [34]:
# !pip install wget
import os
import wget
filepath = drive_path+'/ch10/embeddings/wiki.en.vec'
if not os.path.isfile(filepath):
  wget.download(
      'https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec',
      filepath
  )

100% [....................................................................] 6597238061 / 6597238061

In [36]:
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format(
    filepath,
    binary=False, encoding='utf8'
)

In [40]:
import numpy as np
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# def embed_text(text: str):
#   vector_list = [
#     model.wv[w].reshape(-1, 1) for w in text_to_word_sequence(text)
#     if w in model.wv
#   ]
#   if len(vector_list) > 0:
#     return np.mean(
#         np.concatenate(vector_list, axis=1),
#         axis=1
#     ).reshape(1, 300)
#   else:
#     return np.zeros(shape=(1, 300))


# embed_text('training run').shape

# def embed_text(text: str):
#     vector_list = [
#         model[w].reshape(1, 300)
#         for w in text_to_word_sequence(text)
#         if w in model
#     ]
#     if len(vector_list) > 0:
#         return np.mean(
#             np.concatenate(vector_list, axis=1),
#             axis=1
#         ).reshape(1, 300)
#     else:
#         return np.zeros(shape=(1, 300))

def embed_text(text: str):
    tokens = text_to_word_sequence(text)
    vectors = []

    for w in tokens:
        if w in model:
            vectors.append(model[w])

    if len(vectors) == 0:
        return np.zeros((1, 300))   # guaranteed shape

    vectors = np.vstack(vectors)     # (num_words, 300)
    mean_vec = vectors.mean(axis=0)  # (300,)
    return mean_vec.reshape(1, 300)  # guaranteed shape




In [41]:
train_transformed = np.concatenate(
    [embed_text(t) for t in twenty_train.data]
)

In [42]:
train_transformed.shape

(2257, 300)

In [43]:
rf = RandomForestClassifier().fit(train_transformed, twenty_train.target)

In [44]:
test_transformed = np.concatenate(
    [embed_text(t) for t in twenty_test.data]
)

In [45]:
predicted = rf.predict(test_transformed)
np.mean(predicted == twenty_test.target)

0.8521970705725699

In [46]:
from tensorflow.keras import layers

embedding = layers.Embedding(
    input_dim=5000, 
    output_dim=50, 
    input_length=500
)

In [47]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(twenty_train.data)

In [48]:
X_train = tokenizer.texts_to_sequences(twenty_train.data)
X_test = tokenizer.texts_to_sequences(twenty_test.data)

In [49]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(X_train, padding='post', maxlen=500)
X_test = pad_sequences(X_test, padding='post', maxlen=500)

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras import regularizers

model = Sequential()
model.add(embedding)
model.add(layers.Flatten())
model.add(layers.Dense(
    10,
    activation='relu',
    kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4)
))
model.add(layers.Dense(len(categories), activation='softmax'))
model.compile(optimizer='adam',
              loss=SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 50)           250000    
                                                                 
 flatten (Flatten)           (None, 25000)             0         
                                                                 
 dense (Dense)               (None, 10)                250010    
                                                                 
 dense_1 (Dense)             (None, 4)                 44        
                                                                 
Total params: 500054 (1.91 MB)
Trainable params: 500054 (1.91 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [51]:
model.fit(X_train, twenty_train.target, epochs=10)

Epoch 1/10


71/71 [==============================] - 2s 7ms/step - loss: 1.3376 - accuracy: 0.3797
Epoch 2/10
71/71 [==============================] - 0s 6ms/step - loss: 0.8127 - accuracy: 0.6987
Epoch 3/10
71/71 [==============================] - 0s 6ms/step - loss: 0.2414 - accuracy: 0.9610
Epoch 4/10
71/71 [==============================] - 0s 6ms/step - loss: 0.0967 - accuracy: 0.9960
Epoch 5/10
71/71 [==============================] - 0s 6ms/step - loss: 0.0678 - accuracy: 0.9996
Epoch 6/10
71/71 [==============================] - 0s 6ms/step - loss: 0.0570 - accuracy: 1.0000
Epoch 7/10
71/71 [==============================] - 0s 6ms/step - loss: 0.0507 - accuracy: 1.0000
Epoch 8/10
71/71 [==============================] - 0s 7ms/step - loss: 0.0460 - accuracy: 1.0000
Epoch 9/10
71/71 [==============================] - 0s 7ms/step - loss: 0.0422 - accuracy: 1.0000
Epoch 10/10
71/71 [==============================] - 0s 6ms/step - loss: 0.0391 - accuracy: 1.0000


In [52]:
predicted = model.predict(X_test).argmax(axis=1)
np.mean(predicted == twenty_test.target)

47/47 [==============================] - 0s 5ms/step


0.8901464713715047